In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42) 

## Tasks

### Task 1: Build a Reusable `bootstrap_ci` Function

In [2]:
def bootstrap_ci(data, stat_func=np.mean, n_boot=10_000, ci_level=0.95):
    """
    Returns (lower, upper) percentile bootstrap CI.

    Parameters
    ----------
    data : array-like
        The sample to resample from.
    stat_func : callable
        The statistic to compute on each resample (default: np.mean).
    n_boot : int
        Number of bootstrap resamples.
    ci_level : float
        Confidence level (e.g. 0.95 for a 95 % CI).
    """ 

    data = np.asarray(data)          
    n = len(data)

    resamples = np.random.choice(data, size=(n_boot, n), replace=True)

    boot_stats = np.apply_along_axis(stat_func, axis=1, arr=resamples)

    alpha = (1.0 - ci_level) / 2        
    lower_pct = 100 * alpha             
    upper_pct = 100 * (1.0 - alpha)   

    lower, upper = np.percentile(boot_stats, [lower_pct, upper_pct])

    return (lower, upper) 


data = np.arange(1, 101)

lower, upper = bootstrap_ci(data)
print(f"95% CI for the mean: ({lower:.2f}, {upper:.2f})") 

95% CI for the mean: (44.80, 56.10)


### Task 2: Apply Bootstrap CIs to Real Data

In [3]:
np.random.seed(42)

n = 500  # ≥ 200

df = pd.DataFrame({
    "income": np.random.normal(50000, 10000, n),   # continuous
    "purchased": np.random.binomial(1, 0.3, n)     # binary (0/1)
})

In [4]:
# Mean (continuous)
mean_ci = bootstrap_ci(df["income"], stat_func=np.mean)

# Median (continuous)
median_ci = bootstrap_ci(df["income"], stat_func=np.median)

# Proportion (binary → mean!)
prop_ci = bootstrap_ci(df["purchased"], stat_func=np.mean)

In [5]:
summary = pd.DataFrame({
    "Statistic": ["Mean", "Median", "Proportion"],
    "Column": ["income", "income", "purchased"],
    "CI Lower": [mean_ci[0], median_ci[0], prop_ci[0]],
    "CI Upper": [mean_ci[1], median_ci[1], prop_ci[1]]
})

print(summary)

    Statistic     Column      CI Lower      CI Upper
0        Mean     income  49219.544979  50934.515245
1      Median     income  49019.248003  50986.637318
2  Proportion  purchased      0.258000      0.338050


In [6]:
mean = df["income"].mean()
std = df["income"].std(ddof=1)
n = len(df)

se_mean = std / np.sqrt(n)

mean_ci_norm = st.t.interval(
    confidence=0.95,
    df=n-1,
    loc=mean,
    scale=se_mean
) 

In [7]:
p_hat = df["purchased"].mean()
z = 1.96

se_prop = np.sqrt(p_hat * (1 - p_hat) / n)

prop_ci_norm = (
    p_hat - z * se_prop,
    p_hat + z * se_prop
) 

In [8]:
comparison = pd.DataFrame({
    "Statistic": ["Mean", "Proportion"],
    "Bootstrap Lower": [mean_ci[0], prop_ci[0]],
    "Bootstrap Upper": [mean_ci[1], prop_ci[1]],
    "Normal Lower": [mean_ci_norm[0], prop_ci_norm[0]],
    "Normal Upper": [mean_ci_norm[1], prop_ci_norm[1]]
})

print(comparison)

    Statistic  Bootstrap Lower  Bootstrap Upper  Normal Lower  Normal Upper
0        Mean     49219.544979     50934.515245  49206.198154  50930.561738
1  Proportion         0.258000         0.338050      0.257909      0.338091


### Task 3: Compare Bootstrap CIs with Normal-Approximation CIs

In [9]:
# ── Mean: t-based normal CI ───────────────────────────────────────────────────
mean_val = df["income"].mean()
std_val  = df["income"].std(ddof=1)
n        = len(df)
se_mean  = std_val / np.sqrt(n)

mean_ci_norm = st.t.interval(
    confidence=0.95,
    df=n - 1,
    loc=mean_val,
    scale=se_mean
)
 
# ── Proportion: Wald interval ─────────────────────────────────────────────────
p_hat    = df["purchased"].mean()
z        = 1.96
se_prop  = np.sqrt(p_hat * (1 - p_hat) / n)

prop_ci_norm = (
    p_hat - z * se_prop,
    p_hat + z * se_prop
)

# ── Comparison table ──────────────────────────────────────────────────────────
comparison = pd.DataFrame({
    "Statistic":       ["Mean", "Proportion"],
    "Bootstrap Lower": [mean_ci[0],  prop_ci[0]],
    "Bootstrap Upper": [mean_ci[1],  prop_ci[1]],
    "Normal Lower":    [mean_ci_norm[0], prop_ci_norm[0]],
    "Normal Upper":    [mean_ci_norm[1], prop_ci_norm[1]],
})

comparison = comparison.round(4)
print(comparison.to_string(index=False)) 

 Statistic  Bootstrap Lower  Bootstrap Upper  Normal Lower  Normal Upper
      Mean        49219.545       50934.5152    49206.1982    50930.5617
Proportion            0.258           0.3380        0.2579        0.3381
